# Demo trực tiếp: RPDI-EE trên DeepSeek-R1-Distill-Qwen-1.5B

Nhập một bài toán, mô hình giải bằng **Standard**, **RPDI-EE** và **cắt cứng N token** với cùng ngân sách 768 token, giải mã Greedy.
Code sinh dùng chung `run_one` trong `method_c_rpdi_ee/run_controls.py`, chính là code đã tạo kết quả 200 câu đối chứng.

**Chạy:** Runtime → Change runtime type → **T4 GPU**, rồi chạy lần lượt Mục 1 → 3. Mục 3 in ra một link `*.gradio.live` mở được trên mọi máy.
Mỗi phương pháp mất khoảng 25–40 giây trên T4 khi sinh đủ 768 token.

## 1. Lấy code và cài thư viện

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
REPO_DIR = "/content/slm-reasoning-project"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/luw-cid/slm-reasoning-project.git {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
# Dấu nháy là bắt buộc: không có nháy, shell hiểu '>' là chuyển hướng output
!pip install -q "transformers>=4.48.0" "accelerate>=0.26.0" "gradio>=4.0"

## 2. Nạp mô hình

`load_model_and_tokenizer` tự chọn kiểu số theo GPU: bfloat16 nếu compute capability ≥ 8 (A100, L4), float16 trên T4.

In [ ]:
import sys, time
sys.path.insert(0, REPO_DIR)
import torch
from common.model_loader import load_model_and_tokenizer
from common.metrics import is_answer_correct
from method_c_rpdi_ee.run_controls import run_one

model, tokenizer = load_model_and_tokenizer()
print("dtype:", model.dtype, "| device:", model.device)

## 3. Giao diện demo

- **W, λ**: tham số RPDI-EE (mặc định W=16, λ=1.5 là cấu hình chọn từ quét lưới).
- **N**: số token suy nghĩ của phép cắt cứng (130 ≈ số token suy nghĩ trung bình của RPDI-EE).
- Nhập đáp án đúng (không bắt buộc) để demo tự chấm Đúng/Sai.

In [ ]:
import gradio as gr

EXAMPLES = [
    [
        "Toulouse has twice as many sheep as Charleston. Charleston has 4 times as many sheep as Seattle. How many sheep do Toulouse, Charleston, and Seattle have together if Seattle has 20 sheep?",
        "260"
    ],
    [
        "Julia’s boat sprang a leak while she was out on the lake. The boat was taking on two liters of water for every ten feet she rowed back towards shore. It took her sixteen seconds to row twenty feet. The shore was 64 seconds away. How much water had the boat taken on by the time Julia reached shore?",
        "16"
    ],
    [
        "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
        "18"
    ]
]

METHODS = {"Standard": None, "RPDI-EE": "rpdi", "Cắt cứng": "hardtrunc"}


def solve(question, ground_truth, methods, W, lam, N, max_new_tokens):
    question = (question or "").strip()
    if not question:
        raise gr.Error("Hãy nhập một câu hỏi.")
    rows, texts = [], {}
    for name in ["Standard", "RPDI-EE", "Cắt cứng"]:
        if name not in methods:
            texts[name] = "(không chạy)"
            continue
        cfg = {"Standard": {"type": "standard"},
               "RPDI-EE": {"type": "rpdi", "W": int(W), "lambda_th": float(lam)},
               "Cắt cứng": {"type": "hardtrunc", "N": int(N)}}[name]
        r = run_one(model, tokenizer, question, cfg, int(max_new_tokens), exit_at=None)
        pred = r["predicted_answer"]
        verdict = "—"
        if ground_truth and ground_truth.strip() and pred is not None:
            verdict = "✓ Đúng" if is_answer_correct(pred, ground_truth.strip()) else "✗ Sai"
        elif ground_truth and ground_truth.strip():
            verdict = "✗ Sai"
        if r["total_tokens"] >= int(max_new_tokens):
            budget = "cạn trong pha suy nghĩ" if r["think_tokens"] >= r["total_tokens"] else "cạn trong pha trả lời"
        else:
            budget = "không"
        exit_step = r["exit_step"] if r["early_exited"] else None
        rows.append([name, pred if pred is not None else "—", verdict, r["think_tokens"], r["answer_tokens"],
                     exit_step if exit_step is not None else "—", budget, round(r["latency"], 1)])
        texts[name] = r["response_text"].replace("</think>", "\n\n━━━━━━━━ </think> ━━━━━━━━\n\n")
    return rows, texts["Standard"], texts["RPDI-EE"], texts["Cắt cứng"]


with gr.Blocks(title="Demo RPDI-EE") as demo:
    gr.Markdown("## RPDI-EE: phanh suy luận dựa trên entropy · DeepSeek-R1-Distill-Qwen-1.5B · Greedy")
    with gr.Row():
        with gr.Column(scale=3):
            question = gr.Textbox(label="Câu hỏi toán (tiếng Anh)", lines=4, value=EXAMPLES[0][0])
            ground_truth = gr.Textbox(label="Đáp án đúng (không bắt buộc)", value=EXAMPLES[0][1])
            gr.Examples(EXAMPLES, inputs=[question, ground_truth], label="Câu mẫu GSM8K")
        with gr.Column(scale=2):
            methods = gr.CheckboxGroup(list(METHODS), value=list(METHODS), label="Phương pháp")
            W = gr.Slider(4, 128, value=16, step=1, label="Cửa sổ W")
            lam = gr.Slider(1.05, 3.0, value=1.5, step=0.05, label="Ngưỡng λ")
            N = gr.Slider(16, 512, value=130, step=1, label="Cắt cứng N")
            max_new_tokens = gr.Slider(256, 2048, value=768, step=64, label="Ngân sách token")
            run = gr.Button("Giải", variant="primary")
    table = gr.Dataframe(headers=["Phương pháp", "Đáp án", "Chấm", "Token suy nghĩ", "Token trả lời",
                                  "Bước phanh", "Cạn ngân sách", "Thời gian (s)"], label="Kết quả", interactive=False)
    with gr.Row():
        out_std = gr.Textbox(label="Standard", lines=18, max_lines=30)
        out_rpdi = gr.Textbox(label="RPDI-EE", lines=18, max_lines=30)
        out_ht = gr.Textbox(label="Cắt cứng", lines=18, max_lines=30)
    run.click(solve, [question, ground_truth, methods, W, lam, N, max_new_tokens], [table, out_std, out_rpdi, out_ht])

demo.queue().launch(share=True, debug=False)